# Final OHLCV Collector — Binance.US · Coinbase · Kraken

## Source architecture

| Priority | Source | Used for |
|----------|--------|----------|
| 1 | **Binance.US bulk files** | All pairs, full range, SHA-256 verified |
| 2 | **Binance.US REST API** | Fallback if bulk files missing for a pair/period |
| 3 | **Coinbase** | BTC, ETH, USDT — fills any remaining gaps |
| 4 | **Kraken** | USDC only — fills any remaining gaps (Coinbase has no USDC/USD market) |
| 5 | **NaN** | Dates where no source had data |

**Gap detection is dynamic** — the pipeline detects actual missing dates per pair after each fetch step rather than assuming a fixed suspension window. This handles per-pair differences in when Binance.US stopped/resumed trading.

**No CryptoCompare. No automatic bad-data replacement. No synthetic fills.**

**Output:** `ohlcv_final.xlsx` — one sheet per pair (with Source column) + `Final Data` merged sheet

## Cell 1 — Imports & Configuration

In [1]:
import requests
import hashlib
import zipfile
import io
import time
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

OUTPUT_FILE = "ohlcv_final.xlsx"

START_DT = datetime(2020, 1,  1, tzinfo=timezone.utc)
END_DT   = datetime(2026, 2, 20, 23, 59, 59, tzinfo=timezone.utc)
START_MS = int(START_DT.timestamp() * 1000)
END_MS   = int(END_DT.timestamp()   * 1000)

BINANCE_SYMBOLS  = {"BTC/USD": "BTCUSD",   "ETH/USD": "ETHUSD",
                    "USDC/USD": "USDCUSD",  "USDT/USD": "USDTUSD"}
COINBASE_SYMBOLS = {"BTC/USD": "BTC-USD",   "ETH/USD": "ETH-USD",
                    "USDT/USD": "USDT-USD"}
KRAKEN_SYMBOLS   = {"USDC/USD": "USDCUSD"}

PAIRS            = ["BTC/USD", "ETH/USD", "USDC/USD", "USDT/USD"]
STABLECOIN_PAIRS = {"USDC/USD", "USDT/USD"}

MONTHLY_BASE = "https://data.binance.us/public_data/spot/monthly/klines"
DAILY_BASE   = "https://data.binance.us/public_data/spot/daily/klines"
KLINE_COLS   = {"open_time": 0, "Open": 1, "High": 2,
                "Low": 3,       "Close": 4, "Volume": 5}

STABLECOIN_HIGH      = 1.14
STABLECOIN_LOW       = 0.86
SEAM_PRICE_THRESHOLD = 0.05
SEAM_VOLUME_DAYS     = 30

def all_calendar_dates(start, end):
    dates, cursor = set(), start.replace(hour=0, minute=0, second=0, microsecond=0)
    while cursor <= end:
        dates.add(cursor.strftime("%Y-%m-%d"))
        cursor += timedelta(days=1)
    return dates

EXPECTED_DATES = all_calendar_dates(START_DT, END_DT)

print(f"Range           : {START_DT.strftime('%Y-%m-%d')} → {END_DT.strftime('%Y-%m-%d')}")
print(f"Expected dates  : {len(EXPECTED_DATES)}")
print(f"Stablecoin band : ${STABLECOIN_LOW} – ${STABLECOIN_HIGH}")
print()
print("Gap fill strategy (dynamic — no hardcoded suspension window):")
print("  BTC, ETH, USDT  →  Binance.US bulk  →  Binance.US REST  →  Coinbase")
print("  USDC            →  Binance.US bulk  →  Binance.US REST  →  Kraken")

Range           : 2020-01-01 → 2026-02-20
Expected dates  : 2243
Stablecoin band : $0.86 – $1.14

Gap fill strategy (dynamic — no hardcoded suspension window):
  BTC, ETH, USDT  →  Binance.US bulk  →  Binance.US REST  →  Coinbase
  USDC            →  Binance.US bulk  →  Binance.US REST  →  Kraken


## Cell 2 — Binance.US Fetchers (Bulk + REST fallback)

In [2]:

def _verify_checksum(data_bytes, checksum_url):
    try:
        r = requests.get(checksum_url, timeout=15)
        if r.status_code != 200:
            return True
        expected = r.text.strip().split()[0].lower()
        actual   = hashlib.sha256(data_bytes).hexdigest().lower()
        if actual != expected:
            print(f"    ⚠️  Checksum MISMATCH — skipped")
            return False
        return True
    except Exception:
        return True


def _parse_klines_zip(zip_bytes, source_label):
    rows = []
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        csv_name = [n for n in zf.namelist() if n.endswith(".csv")][0]
        with zf.open(csv_name) as f:
            for line in f:
                parts = line.decode().strip().split(",")
                if not parts or not parts[0].isdigit():
                    continue
                rows.append({
                    "Date":   datetime.utcfromtimestamp(
                                int(parts[KLINE_COLS["open_time"]]) / 1000
                              ).strftime("%Y-%m-%d"),
                    "Open":   float(parts[KLINE_COLS["Open"]]),
                    "High":   float(parts[KLINE_COLS["High"]]),
                    "Low":    float(parts[KLINE_COLS["Low"]]),
                    "Close":  float(parts[KLINE_COLS["Close"]]),
                    "Volume": float(parts[KLINE_COLS["Volume"]]),
                    "Source": source_label,
                })
    if not rows:
        return pd.DataFrame(columns=["Date","Open","High","Low","Close","Volume","Source"])
    return pd.DataFrame(rows)


def _download_zip(url, checksum_url):
    r = requests.get(url, timeout=60)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.content if _verify_checksum(r.content, checksum_url) else None


def fetch_binance_bulk(pair):
    """
    Download all available Binance.US bulk kline files for a pair.
    Monthly first; daily fallback on 404.
    Returns DataFrame or empty DataFrame if nothing available.
    """
    symbol = BINANCE_SYMBOLS[pair]
    frames = []
    cursor = START_DT.replace(day=1, hour=0, minute=0, second=0)
    today  = datetime.utcnow().replace(tzinfo=timezone.utc)
    print(f"  [Binance.US bulk] {pair} ({symbol})")

    while cursor <= END_DT:
        ym         = cursor.strftime("%Y-%m")
        is_current = (cursor.year == today.year and cursor.month == today.month)
        next_month = (cursor.replace(day=28) + timedelta(days=4)).replace(day=1)

        if not is_current:
            url  = f"{MONTHLY_BASE}/{symbol}/1d/{symbol}-1d-{ym}.zip"
            data = _download_zip(url, url + ".CHECKSUM")
            if data is not None:
                df = _parse_klines_zip(data, "Binance.US bulk")
                df = df[(df["Date"] >= START_DT.strftime("%Y-%m-%d")) &
                        (df["Date"] <= END_DT.strftime("%Y-%m-%d"))]
                if not df.empty:
                    frames.append(df)
                    print(f"    {ym}  ✅ monthly ({len(df)} rows)")
                    cursor = next_month
                    continue

        print(f"    {ym}  → daily fallback...")
        day_cur, daily_count = cursor, 0
        while day_cur < next_month and day_cur <= END_DT:
            ymd  = day_cur.strftime("%Y-%m-%d")
            url  = f"{DAILY_BASE}/{symbol}/1d/{symbol}-1d-{ymd}.zip"
            data = _download_zip(url, url + ".CHECKSUM")
            if data is not None:
                df = _parse_klines_zip(data, "Binance.US bulk")
                if not df.empty:
                    frames.append(df)
                    daily_count += 1
            day_cur += timedelta(days=1)
            time.sleep(0.05)
        print(f"    {ym}  {'✅ ' + str(daily_count) + ' day files' if daily_count else '⚠️  no data'}")
        cursor = next_month

    if not frames:
        print(f"  [Binance.US bulk] ⚠️  No bulk files found for {pair}")
        return pd.DataFrame(columns=["Date","Open","High","Low","Close","Volume","Source"])

    combined = (pd.concat(frames, ignore_index=True)
                  .drop_duplicates("Date").sort_values("Date").reset_index(drop=True))
    print(f"  [Binance.US bulk] ✅ {len(combined)} rows "
          f"({combined['Date'].iloc[0]} → {combined['Date'].iloc[-1]})")
    return combined


def fetch_binance_rest_for_gaps(pair, missing_dates):
    """
    Fetch specific missing dates from Binance.US REST API (/api/v3/klines).
    Groups missing dates into contiguous ranges for efficient pagination.
    Used as a fallback when bulk files are absent for a pair or period.
    """
    if not missing_dates:
        return pd.DataFrame(columns=["Date","Open","High","Low","Close","Volume","Source"])

    symbol = BINANCE_SYMBOLS[pair]
    url    = "https://api.binance.us/api/v3/klines"

    sorted_dates = sorted(missing_dates)
    ranges, rs, prev = [], sorted_dates[0], sorted_dates[0]
    for d in sorted_dates[1:]:
        if (datetime.strptime(d,"%Y-%m-%d") -
            datetime.strptime(prev,"%Y-%m-%d")).days > 1:
            ranges.append((rs, prev)); rs = d
        prev = d
    ranges.append((rs, prev))

    print(f"  [Binance.US REST] {pair} — {len(missing_dates)} gaps → "
          f"{len(ranges)} range(s)")

    frames = []
    for r_start, r_end in ranges:
        start_ms = int(datetime.strptime(r_start,"%Y-%m-%d")
                       .replace(tzinfo=timezone.utc).timestamp() * 1000)
        end_ms   = int(datetime.strptime(r_end,"%Y-%m-%d")
                       .replace(hour=23,minute=59,second=59,tzinfo=timezone.utc)
                       .timestamp() * 1000)
        rows, cursor = [], start_ms
        print(f"    → {r_start} to {r_end}")

        while cursor <= end_ms:
            try:
                r = requests.get(url, params={
                    "symbol": symbol, "interval": "1d",
                    "startTime": cursor, "endTime": end_ms, "limit": 1000
                }, timeout=30)
                r.raise_for_status()
                data = r.json()
            except Exception as e:
                print(f"    ⚠️  REST fetch failed for {r_start}→{r_end}: {e}")
                break
            if not data or isinstance(data, dict):
                break
            for k in data:
                rows.append({
                    "Date":   datetime.utcfromtimestamp(k[0]/1000).strftime("%Y-%m-%d"),
                    "Open":   float(k[1]), "High":  float(k[2]),
                    "Low":    float(k[3]), "Close": float(k[4]),
                    "Volume": float(k[5]),
                    "Source": "Binance.US REST",
                })
            last_ts = data[-1][0]
            if last_ts >= end_ms or len(data) < 1000:
                break
            cursor = last_ts + 86_400_000
            time.sleep(0.25)

        if rows:
            frames.append(pd.DataFrame(rows))

    if not frames:
        return pd.DataFrame(columns=["Date","Open","High","Low","Close","Volume","Source"])

    combined = (pd.concat(frames, ignore_index=True)
                  .drop_duplicates("Date").sort_values("Date").reset_index(drop=True))
    filled = combined[combined["Date"].isin(missing_dates)].reset_index(drop=True)
    print(f"  [Binance.US REST] ✅ {len(filled)} of {len(missing_dates)} dates filled")
    return filled


print("✅ Binance.US fetchers ready (bulk + REST fallback).")

✅ Binance.US fetchers ready (bulk + REST fallback).


## Cell 3 — Coinbase Fetcher

In [3]:
def fetch_coinbase_for_gaps(pair, missing_dates):
    """
    Fetch specific missing dates from Coinbase Exchange API.
    Groups missing dates into contiguous ranges (300-day max per request).
    Used for BTC, ETH, USDT — NOT available for USDC.
    Coinbase candle format: [time, low, high, open, close, volume]
    """
    if not missing_dates or pair not in COINBASE_SYMBOLS:
        return pd.DataFrame(columns=["Date","Open","High","Low","Close","Volume","Source"])

    symbol     = COINBASE_SYMBOLS[pair]
    url        = f"https://api.exchange.coinbase.com/products/{symbol}/candles"
    chunk_secs = 300 * 86400

    sorted_dates = sorted(missing_dates)
    ranges, rs, prev = [], sorted_dates[0], sorted_dates[0]
    for d in sorted_dates[1:]:
        if (datetime.strptime(d,"%Y-%m-%d") -
            datetime.strptime(prev,"%Y-%m-%d")).days > 1:
            ranges.append((rs, prev)); rs = d
        prev = d
    ranges.append((rs, prev))

    print(f"  [Coinbase] {pair} ({symbol}) — {len(missing_dates)} gaps → "
          f"{len(ranges)} range(s)")

    frames = []
    for r_start, r_end in ranges:
        print(f"    → {r_start} to {r_end}")
        rows   = []
        cursor = int(datetime.strptime(r_start,"%Y-%m-%d")
                     .replace(tzinfo=timezone.utc).timestamp())
        end_ts = int(datetime.strptime(r_end,"%Y-%m-%d")
                     .replace(hour=23,minute=59,second=59,tzinfo=timezone.utc)
                     .timestamp())

        while cursor <= end_ts:
            chunk_end = min(cursor + chunk_secs, end_ts)
            try:
                r = requests.get(url, params={
                    "granularity": 86400,
                    "start": datetime.utcfromtimestamp(cursor).strftime("%Y-%m-%dT%H:%M:%SZ"),
                    "end":   datetime.utcfromtimestamp(chunk_end).strftime("%Y-%m-%dT%H:%M:%SZ"),
                }, timeout=30)
                r.raise_for_status()
                data = r.json()
            except Exception as e:
                print(f"    ⚠️  Coinbase fetch failed: {e}")
                break

            if data and not isinstance(data, dict):
                for k in data:
                    rows.append({
                        "Date":   datetime.utcfromtimestamp(k[0]).strftime("%Y-%m-%d"),
                        "Open":   float(k[3]), "High":  float(k[2]),
                        "Low":    float(k[1]), "Close": float(k[4]),
                        "Volume": float(k[5]), "Source": "Coinbase",
                    })
            cursor = chunk_end + 86400
            time.sleep(0.3)

        if rows:
            frames.append(pd.DataFrame(rows))

    if not frames:
        print(f"  [Coinbase] ⚠️  No data returned")
        return pd.DataFrame(columns=["Date","Open","High","Low","Close","Volume","Source"])

    combined = (pd.concat(frames, ignore_index=True)
                  .drop_duplicates("Date").sort_values("Date").reset_index(drop=True))
    filled = combined[combined["Date"].isin(missing_dates)].reset_index(drop=True)
    print(f"  [Coinbase] ✅ {len(filled)} of {len(missing_dates)} dates filled")
    return filled


print("✅ Coinbase fetcher ready.")

✅ Coinbase fetcher ready.


## Cell 4 — Kraken Fetcher

In [4]:
def fetch_kraken_for_gaps(pair, missing_dates):
    """
    Fetch specific missing dates from Kraken public REST API.
    Used for USDC/USD only.
    Kraken paginates backwards via the 'since' Unix timestamp.
    Kraken OHLC format: [time, open, high, low, close, vwap, volume, count]
    """
    if not missing_dates or pair not in KRAKEN_SYMBOLS:
        return pd.DataFrame(columns=["Date","Open","High","Low","Close","Volume","Source"])

    symbol       = KRAKEN_SYMBOLS[pair]
    url          = "https://api.kraken.com/0/public/OHLC"
    sorted_dates = sorted(missing_dates)

    print(f"  [Kraken] {pair} ({symbol}) — {len(missing_dates)} gaps")

    since  = int(datetime.strptime(sorted_dates[0], "%Y-%m-%d")
                 .replace(tzinfo=timezone.utc).timestamp())
    end_ts = int(datetime.strptime(sorted_dates[-1], "%Y-%m-%d")
                 .replace(hour=23,minute=59,second=59,tzinfo=timezone.utc)
                 .timestamp())
    rows   = []

    while True:
        try:
            r = requests.get(url, params={
                "pair": symbol, "interval": 1440, "since": since
            }, timeout=30)
            r.raise_for_status()
            resp = r.json()
        except Exception as e:
            print(f"  [Kraken] ⚠️  Fetch failed: {e}")
            break

        if resp.get("error"):
            print(f"  [Kraken] ⚠️  API error: {resp['error']}")
            break

        result   = resp["result"]
        last     = int(result.get("last", 0))
        pair_key = [k for k in result if k != "last"][0]
        data     = result[pair_key]

        if not data:
            break

        new_rows = 0
        for k in data:
            ts = int(k[0])
            if ts > end_ts:
                break
            rows.append({
                "Date":   datetime.utcfromtimestamp(ts).strftime("%Y-%m-%d"),
                "Open":   float(k[1]), "High":  float(k[2]),
                "Low":    float(k[3]), "Close": float(k[4]),
                "Volume": float(k[6]), "Source": "Kraken",
            })
            new_rows += 1

        if not last or last <= since or new_rows == 0:
            break
        since = last
        time.sleep(0.5)

    if not rows:
        print(f"  [Kraken] ⚠️  No data returned")
        return pd.DataFrame(columns=["Date","Open","High","Low","Close","Volume","Source"])

    combined = (pd.DataFrame(rows)
                  .drop_duplicates("Date").sort_values("Date").reset_index(drop=True))
    filled = combined[combined["Date"].isin(missing_dates)].reset_index(drop=True)
    print(f"  [Kraken] ✅ {len(filled)} of {len(missing_dates)} dates filled")
    return filled


print("✅ Kraken fetcher ready.")

✅ Kraken fetcher ready.


## Cell 5 — Collection Pipeline

In [5]:
def merge_sources(*dfs):
    """
    Concatenate DataFrames in priority order (first = highest priority).
    On duplicate dates, keeps the first occurrence.
    """
    non_empty = [df for df in dfs if not df.empty]
    if not non_empty:
        return pd.DataFrame(columns=["Date","Open","High","Low","Close","Volume","Source"])
    return (pd.concat(non_empty, ignore_index=True)
              .drop_duplicates("Date", keep="first")
              .sort_values("Date")
              .reset_index(drop=True))


def collect_pair(pair):
    """
    Full collection pipeline for one pair.

    Step 1  — Binance.US bulk files (primary, SHA-256 verified)
    Step 2  — Binance.US REST API fills dates still missing after bulk
              (handles pairs/periods where bulk files don't exist)
    Step 3  — Coinbase (BTC/ETH/USDT) or Kraken (USDC) fills any
              remaining gaps that Binance.US cannot cover at all

    Gap detection is dynamic at each step — no hardcoded date windows.
    Binance.US (bulk or REST) always takes priority over Coinbase/Kraken.
    """
    print(f"\n{'═'*65}\n  {pair}\n{'═'*65}")

    try:
        bulk_df = fetch_binance_bulk(pair)
    except Exception as e:
        print(f"  [Binance.US bulk] ❌ {e}")
        bulk_df = pd.DataFrame(
            columns=["Date","Open","High","Low","Close","Volume","Source"])

    missing_after_bulk = EXPECTED_DATES - set(bulk_df["Date"])
    print(f"  After bulk   : {len(bulk_df)} rows, "
          f"{len(missing_after_bulk)} dates missing")

    try:
        rest_df = fetch_binance_rest_for_gaps(pair, missing_after_bulk)
    except Exception as e:
        print(f"  [Binance.US REST] ❌ {e}")
        rest_df = pd.DataFrame(
            columns=["Date","Open","High","Low","Close","Volume","Source"])

    combined        = merge_sources(bulk_df, rest_df)
    missing_after_rest = EXPECTED_DATES - set(combined["Date"])
    print(f"  After REST   : {len(combined)} rows, "
          f"{len(missing_after_rest)} dates missing")

    if missing_after_rest:
        if pair == "USDC/USD":
            try:
                third_df = fetch_kraken_for_gaps(pair, missing_after_rest)
            except Exception as e:
                print(f"  [Kraken] ❌ {e}")
                third_df = pd.DataFrame(
                    columns=["Date","Open","High","Low","Close","Volume","Source"])
        else:
            try:
                third_df = fetch_coinbase_for_gaps(pair, missing_after_rest)
            except Exception as e:
                print(f"  [Coinbase] ❌ {e}")
                third_df = pd.DataFrame(
                    columns=["Date","Open","High","Low","Close","Volume","Source"])

        combined = merge_sources(combined, third_df)

    still_missing = EXPECTED_DATES - set(combined["Date"])
    src_counts    = combined["Source"].value_counts()

    print(f"\n  ── Final ──")
    print(f"  Total rows     : {len(combined)}")
    print(f"  Still missing  : {len(still_missing)} dates (will be NaN in merged sheet)")
    for src, cnt in src_counts.items():
        src_df = combined[combined["Source"] == src]
        print(f"    {src:<25} {cnt:>5} rows "
              f"({src_df['Date'].min()} → {src_df['Date'].max()})")

    return combined


all_data = {}
for pair in PAIRS:
    all_data[pair] = collect_pair(pair)


═════════════════════════════════════════════════════════════════
  BTC/USD
═════════════════════════════════════════════════════════════════
  [Binance.US bulk] BTC/USD (BTCUSD)
    2020-01  ✅ monthly (31 rows)
    2020-02  ✅ monthly (29 rows)


C:\Users\paliw\AppData\Local\Temp\ipykernel_27436\3952285554.py:60: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today  = datetime.utcnow().replace(tzinfo=timezone.utc)
C:\Users\paliw\AppData\Local\Temp\ipykernel_27436\3952285554.py:28: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "Date":   datetime.utcfromtimestamp(


    2020-03  ✅ monthly (31 rows)
    2020-04  ✅ monthly (30 rows)
    2020-05  ✅ monthly (31 rows)
    2020-06  ✅ monthly (30 rows)
    2020-07  ✅ monthly (31 rows)
    2020-08  ✅ monthly (31 rows)
    2020-09  ✅ monthly (30 rows)
    2020-10  ✅ monthly (31 rows)
    2020-11  ✅ monthly (30 rows)
    2020-12  ✅ monthly (31 rows)
    2021-01  ✅ monthly (31 rows)
    2021-02  ✅ monthly (28 rows)
    2021-03  ✅ monthly (31 rows)
    2021-04  ✅ monthly (30 rows)
    2021-05  ✅ monthly (31 rows)
    2021-06  ✅ monthly (30 rows)
    2021-07  ✅ monthly (31 rows)
    2021-08  ✅ monthly (31 rows)
    2021-09  ✅ monthly (30 rows)
    2021-10  ✅ monthly (31 rows)
    2021-11  ✅ monthly (30 rows)
    2021-12  ✅ monthly (31 rows)
    2022-01  ✅ monthly (31 rows)
    2022-02  ✅ monthly (28 rows)
    2022-03  ✅ monthly (31 rows)
    2022-04  ✅ monthly (30 rows)
    2022-05  ✅ monthly (31 rows)
    2022-06  ✅ monthly (30 rows)
    2022-07  ✅ monthly (31 rows)
    2022-08  ✅ monthly (31 rows)
    2022-0

C:\Users\paliw\AppData\Local\Temp\ipykernel_27436\3003817540.py:43: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "start": datetime.utcfromtimestamp(cursor).strftime("%Y-%m-%dT%H:%M:%SZ"),
C:\Users\paliw\AppData\Local\Temp\ipykernel_27436\3003817540.py:44: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "end":   datetime.utcfromtimestamp(chunk_end).strftime("%Y-%m-%dT%H:%M:%SZ"),
C:\Users\paliw\AppData\Local\Temp\ipykernel_27436\3003817540.py:55: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in 

  [Coinbase] ✅ 585 of 585 dates filled

  ── Final ──
  Total rows     : 2243
  Still missing  : 0 dates (will be NaN in merged sheet)
    Binance.US bulk            1658 rows (2020-01-01 → 2026-02-20)
    Coinbase                    585 rows (2023-07-15 → 2025-02-18)

═════════════════════════════════════════════════════════════════
  ETH/USD
═════════════════════════════════════════════════════════════════
  [Binance.US bulk] ETH/USD (ETHUSD)
    2020-01  ✅ monthly (31 rows)
    2020-02  ✅ monthly (29 rows)


...

C:\Users\paliw\AppData\Local\Temp\ipykernel_27436\1996503245.py:54: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "Date":   datetime.utcfromtimestamp(ts).strftime("%Y-%m-%d"),


  [Kraken] ✅ 349 of 791 dates filled

  ── Final ──
  Total rows     : 1801
  Still missing  : 442 dates (will be NaN in merged sheet)
    Binance.US bulk            1452 rows (2020-07-24 → 2026-02-20)
    Kraken                      349 rows (2024-03-08 → 2025-02-19)

═════════════════════════════════════════════════════════════════
  USDT/USD
═════════════════════════════════════════════════════════════════
  [Binance.US bulk] USDT/USD (USDTUSD)
    2020-01  ✅ monthly (31 rows)
    2020-02  ✅ monthly (29 rows)


C:\Users\paliw\AppData\Local\Temp\ipykernel_27436\3952285554.py:60: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today  = datetime.utcnow().replace(tzinfo=timezone.utc)
C:\Users\paliw\AppData\Local\Temp\ipykernel_27436\3952285554.py:28: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "Date":   datetime.utcfromtimestamp(


    2020-03  ✅ monthly (31 rows)
    2020-04  ✅ monthly (30 rows)
    2020-05  ✅ monthly (31 rows)
    2020-06  ✅ monthly (30 rows)
    2020-07  ✅ monthly (31 rows)
    2020-08  ✅ monthly (31 rows)
    2020-09  ✅ monthly (30 rows)
    2020-10  ✅ monthly (31 rows)
    2020-11  ✅ monthly (30 rows)
    2020-12  ✅ monthly (31 rows)
    2021-01  ✅ monthly (31 rows)
    2021-02  ✅ monthly (28 rows)
    2021-03  ✅ monthly (31 rows)
    2021-04  ✅ monthly (30 rows)
    2021-05  ✅ monthly (31 rows)
    2021-06  ✅ monthly (30 rows)
    2021-07  ✅ monthly (31 rows)
    2021-08  ✅ monthly (31 rows)
    2021-09  ✅ monthly (30 rows)
    2021-10  ✅ monthly (31 rows)
    2021-11  ✅ monthly (30 rows)
    2021-12  ✅ monthly (31 rows)
    2022-01  ✅ monthly (31 rows)
    2022-02  ✅ monthly (28 rows)
    2022-03  ✅ monthly (31 rows)
    2022-04  ✅ monthly (30 rows)
    2022-05  ✅ monthly (31 rows)
    2022-06  ✅ monthly (30 rows)
    2022-07  ✅ monthly (31 rows)
    2022-08  ✅ monthly (31 rows)
    2022-0

C:\Users\paliw\AppData\Local\Temp\ipykernel_27436\3952285554.py:161: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "Date":   datetime.utcfromtimestamp(k[0]/1000).strftime("%Y-%m-%d"),


## Cell 6 — Checks & Validation

In [6]:
print("═" * 70)
print("  VALIDATION REPORT")
print("═" * 70)

all_checks_passed = True

for pair, df in all_data.items():
    print(f"\n{'─'*70}")
    print(f"  {pair}")
    print(f"{'─'*70}")

    if df.empty:
        print("  ❌ No data — all checks skipped")
        all_checks_passed = False
        continue

    dupes = df[df.duplicated("Date", keep=False)]
    if dupes.empty:
        print("  ✅ Check 1 — No duplicate dates")
    else:
        print(f"  ❌ Check 1 — {len(dupes)} duplicate date rows:")
        print(dupes[["Date","Source"]].to_string(index=False))
        all_checks_passed = False

    covered = set(df["Date"])
    missing = sorted(EXPECTED_DATES - covered)
    if not missing:
        print("  ✅ Check 2 — Full calendar coverage")
    else:
        ranges, rs, prev = [], missing[0], missing[0]
        for d in missing[1:]:
            if (datetime.strptime(d,"%Y-%m-%d") -
                datetime.strptime(prev,"%Y-%m-%d")).days > 1:
                ranges.append((rs, prev)); rs = d
            prev = d
        ranges.append((rs, prev))
        print(f"  ⚠️  Check 2 — {len(missing)} missing dates in "
              f"{len(ranges)} gap(s):")
        for r_start, r_end in ranges:
            n = (datetime.strptime(r_end,"%Y-%m-%d") -
                 datetime.strptime(r_start,"%Y-%m-%d")).days + 1
            prev_src = df[df["Date"] < r_start]["Source"].iloc[-1] \
                       if not df[df["Date"] < r_start].empty else "none"
            next_src = df[df["Date"] > r_end]["Source"].iloc[0] \
                       if not df[df["Date"] > r_end].empty else "none"
            print(f"       {r_start} → {r_end}  ({n} days)  "
                  f"[before: {prev_src} | after: {next_src}]")

    print(f"  ✅ Check 3 — Source attribution:")
    for src in df["Source"].unique():
        s = df[df["Source"] == src]
        print(f"       {src:<30} {len(s):>5} rows  "
              f"({s['Date'].min()} → {s['Date'].max()})")

    if pair in STABLECOIN_PAIRS:
        bad = df[(df["High"] > STABLECOIN_HIGH) | (df["Low"] < STABLECOIN_LOW)]
        if bad.empty:
            print(f"  ✅ Check 4 — No values outside "
                  f"[${STABLECOIN_LOW}, ${STABLECOIN_HIGH}]")
        else:
            print(f"  ⚠️  Check 4 — {len(bad)} rows outside peg band:")
            print(f"       {'Date':<12} {'High':>10} {'Low':>10} "
                  f"{'Close':>10}  Source")
            for _, row in bad.iterrows():
                fh = " ←" if row["High"] > STABLECOIN_HIGH else ""
                fl = " ←" if row["Low"]  < STABLECOIN_LOW  else ""
                print(f"       {row['Date']:<12} "
                      f"{row['High']:>10.4f}{fh}  "
                      f"{row['Low']:>10.4f}{fl}  "
                      f"{row['Close']:>10.4f}  {row['Source']}")
    else:
        print("  ─  Check 4 — Non-stablecoin, skipped")

    df_s       = df.sort_values("Date").reset_index(drop=True)
    seam_flags = []
    for i in range(1, len(df_s)):
        if df_s.loc[i,"Source"] != df_s.loc[i-1,"Source"]:
            prev_close = df_s.loc[i-1,"Close"]
            curr_open  = df_s.loc[i,  "Open"]
            pct        = abs(curr_open - prev_close) / prev_close \
                         if prev_close > 0 else 0
            seam_flags.append({
                "flag":         "⚠️ " if pct > SEAM_PRICE_THRESHOLD else "✅",
                "date_before":  df_s.loc[i-1,"Date"],
                "src_before":   df_s.loc[i-1,"Source"],
                "close_before": prev_close,
                "date_after":   df_s.loc[i,  "Date"],
                "src_after":    df_s.loc[i,  "Source"],
                "open_after":   curr_open,
                "pct":          pct,
            })

    if not seam_flags:
        print("  ✅ Check 5 — No source transitions")
    else:
        print(f"  Check 5 — {len(seam_flags)} source transition(s):")
        for s in seam_flags:
            print(f"       {s['flag']}  "
                  f"{s['date_before']} [{s['src_before']}] "
                  f"Close={s['close_before']:.4f}  →  "
                  f"{s['date_after']} [{s['src_after']}] "
                  f"Open={s['open_after']:.4f}  Δ={s['pct']*100:.2f}%")
            if s['flag'] == "⚠️ ":
                all_checks_passed = False

    if seam_flags:
        print(f"  Check 6 — Volume scale at seams "
              f"(±{SEAM_VOLUME_DAYS}-day avg):")
        for s in seam_flags:
            before = df_s[df_s["Date"] < s["date_after"]] \
                         .tail(SEAM_VOLUME_DAYS)["Volume"].mean()
            after  = df_s[df_s["Date"] >= s["date_after"]] \
                         .head(SEAM_VOLUME_DAYS)["Volume"].mean()
            ratio  = after / before if before > 0 else float("nan")
            flag   = "⚠️ " if (ratio > 5 or ratio < 0.2) else "✅"
            print(f"       {flag}  Seam {s['date_after']}:  "
                  f"before={before:>14,.2f}  "
                  f"after={after:>14,.2f}  ratio={ratio:.2f}x")
    else:
        print("  ─  Check 6 — No seams to check")

print(f"\n{'═'*70}")
print("  ✅ ALL CHECKS PASSED" if all_checks_passed
      else "  ⚠️  SOME FLAGS — review output above")
print("═" * 70)

══════════════════════════════════════════════════════════════════════
  VALIDATION REPORT
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
  BTC/USD
──────────────────────────────────────────────────────────────────────
  ✅ Check 1 — No duplicate dates
  ✅ Check 2 — Full calendar coverage
  ✅ Check 3 — Source attribution:
       Binance.US bulk                 1658 rows  (2020-01-01 → 2026-02-20)
       Coinbase                         585 rows  (2023-07-15 → 2025-02-18)
  ─  Check 4 — Non-stablecoin, skipped
  Check 5 — 2 source transition(s):
       ⚠️   2023-07-14 [Binance.US bulk] Close=25073.2100  →  2023-07-15 [Coinbase] Open=30330.9300  Δ=20.97%
       ✅  2025-02-18 [Coinbase] Close=95607.4000  →  2025-02-19 [Binance.US bulk] Open=96236.2800  Δ=0.66%
  Check 6 — Volume scale at seams (±30-day avg):
       ⚠️   Seam 2023-07-15:  before=        141.05  after=      7,846.39  ratio=55.63x


## Cell 7 — Collection Summary

In [7]:
print("═" * 80)
print("  FINAL COLLECTION SUMMARY")
print("═" * 80)
print(f"  {'Pair':<12} {'Rows':>5}  {'Missing':>8}  {'Null%':>6}  "
      f"{'First date':<13}  Last date")
print("─" * 80)
for pair, df in all_data.items():
    if df.empty:
        print(f"  {pair:<12}  ❌ No data")
        continue
    n_miss   = len(EXPECTED_DATES - set(df["Date"]))
    null_pct = df[["Open","High","Low","Close","Volume"]].isnull().mean().mean() * 100
    print(f"  {pair:<12} {len(df):>5}  {n_miss:>8}  "
          f"{null_pct:>5.1f}%  "
          f"{df['Date'].iloc[0]:<13}  {df['Date'].iloc[-1]}")
print("─" * 80)
print()
print("  Rows by source:")
for pair, df in all_data.items():
    if df.empty: continue
    print(f"  {pair}")
    for src, cnt in df["Source"].value_counts().items():
        s = df[df["Source"] == src]
        print(f"    {src:<30} {cnt:>5} rows  "
              f"({s['Date'].min()} → {s['Date'].max()})")
print("═" * 80)

════════════════════════════════════════════════════════════════════════════════
  FINAL COLLECTION SUMMARY
════════════════════════════════════════════════════════════════════════════════
  Pair          Rows   Missing   Null%  First date     Last date
────────────────────────────────────────────────────────────────────────────────
  BTC/USD       2243         0    0.0%  2020-01-01     2026-02-20
  ETH/USD       2243         0    0.0%  2020-01-01     2026-02-20
  USDC/USD      1801       442    0.0%  2020-07-24     2026-02-20
  USDT/USD      2243         0    0.0%  2020-01-01     2026-02-20
────────────────────────────────────────────────────────────────────────────────

  Rows by source:
  BTC/USD
    Binance.US bulk                 1658 rows  (2020-01-01 → 2026-02-20)
    Coinbase                         585 rows  (2023-07-15 → 2025-02-18)
  ETH/USD
    Binance.US bulk                 1658 rows  (2020-01-01 → 2026-02-20)
    Coinbase                         585 rows  (2023-07-15 → 2

## Cell 8 — Wide-Merge & Export to Excel

In [8]:

header_font   = Font(bold=True, color="FFFFFF", name="Arial", size=10)
header_fill   = PatternFill("solid", start_color="1F4E79")
alt_fill      = PatternFill("solid", start_color="D6E4F0")
coinbase_fill = PatternFill("solid", start_color="E2EFDA")
kraken_fill   = PatternFill("solid", start_color="FFF2CC")
rest_fill     = PatternFill("solid", start_color="FCE4D6")
thin          = Border(left=Side(style="thin"),  right=Side(style="thin"),
                       top=Side(style="thin"),   bottom=Side(style="thin"))
center        = Alignment(horizontal="center")
SOURCE_FILL   = {"Coinbase": coinbase_fill,
                 "Kraken":   kraken_fill,
                 "Binance.US REST": rest_fill}

wb = Workbook()
wb.remove(wb.active)

ws_leg = wb.create_sheet(title="Legend", index=0)
legend_rows = [
    ("Colour",        "Source",           "Used for"),
    ("Blue stripe",   "Binance.US bulk",   "Primary — checksum-verified bulk files"),
    ("Orange",        "Binance.US REST",   "Fallback where bulk files absent"),
    ("Green",         "Coinbase",          "BTC/ETH/USDT gap fill"),
    ("Yellow",        "Kraken",            "USDC/USD gap fill only"),
    ("White",         "(Final Data sheet)","Source columns removed; NaN where no source had data"),
]
for r, row in enumerate(legend_rows, 1):
    for c, val in enumerate(row, 1):
        cell = ws_leg.cell(row=r, column=c, value=val)
        cell.font = Font(bold=(r == 1), name="Arial", size=10)
ws_leg.column_dimensions["A"].width = 16
ws_leg.column_dimensions["B"].width = 20
ws_leg.column_dimensions["C"].width = 55

PER_HEADERS = ["Date","Open","High","Low","Close","Volume","Source"]
PER_WIDTHS  = [14, 18, 18, 18, 18, 20, 28]

for label, df in all_data.items():
    ws = wb.create_sheet(title=label.replace("/","_"))
    if df.empty:
        continue
    for c_idx, (h, w) in enumerate(zip(PER_HEADERS, PER_WIDTHS), 1):
        cell = ws.cell(row=1, column=c_idx, value=h)
        cell.font = header_font; cell.fill = header_fill
        cell.alignment = center; cell.border = thin
        ws.column_dimensions[get_column_letter(c_idx)].width = w
    ws.row_dimensions[1].height = 18
    for r_idx, row in enumerate(df.itertuples(index=False), 2):
        rd   = row._asdict()
        src  = rd.get("Source","")
        fill = SOURCE_FILL.get(src, alt_fill if r_idx % 2 == 0 else None)
        for c_idx, h in enumerate(PER_HEADERS, 1):
            val       = rd.get(h)
            write_val = None if (isinstance(val, float) and pd.isna(val)) else val
            cell = ws.cell(row=r_idx, column=c_idx, value=write_val)
            cell.font = Font(name="Arial", size=10)
            cell.border = thin; cell.alignment = center
            if fill: cell.fill = fill
            if h == "Date":     cell.number_format = "YYYY-MM-DD"
            elif h == "Volume": cell.number_format = "#,##0.00"
            elif h == "Source": pass
            else:               cell.number_format = "#,##0.00000000"
    ws.freeze_panes = "A2"
    print(f"  Sheet '{label}' written — {len(df)} rows")

merged_df = None
for label, df in all_data.items():
    if df.empty: continue
    prefix  = label.replace("/","_")
    renamed = df.drop(columns=["Source"]).rename(columns={
        "Open":   f"{prefix}_Open",  "High":   f"{prefix}_High",
        "Low":    f"{prefix}_Low",   "Close":  f"{prefix}_Close",
        "Volume": f"{prefix}_Volume",
    })
    merged_df = renamed if merged_df is None else pd.merge(
        merged_df, renamed, on="Date", how="outer")

merged_df = merged_df.sort_values("Date").reset_index(drop=True)
ws_m   = wb.create_sheet(title="Final Data")
m_cols = list(merged_df.columns)

for c_idx, col in enumerate(m_cols, 1):
    cell = ws_m.cell(row=1, column=c_idx, value=col)
    cell.font = header_font; cell.fill = header_fill
    cell.alignment = center; cell.border = thin
    ws_m.column_dimensions[get_column_letter(c_idx)].width = 14 if col=="Date" else 20
ws_m.row_dimensions[1].height = 18

for r_idx, row in enumerate(merged_df.itertuples(index=False), 2):
    fill = alt_fill if r_idx % 2 == 0 else None
    for c_idx, (col, val) in enumerate(zip(m_cols, row), 1):
        write_val = None if (isinstance(val, float) and pd.isna(val)) else val
        cell = ws_m.cell(row=r_idx, column=c_idx, value=write_val)
        cell.font = Font(name="Arial", size=10)
        cell.border = thin; cell.alignment = center
        if fill: cell.fill = fill
        if col == "Date":      cell.number_format = "YYYY-MM-DD"
        elif "Volume" in col:  cell.number_format = "#,##0.00"
        else:                  cell.number_format = "#,##0.00000000"
ws_m.freeze_panes = "B2"

wb.save(OUTPUT_FILE)

print(f"\nMerged shape : {merged_df.shape}")
print(f"Date range   : {merged_df['Date'].iloc[0]} → {merged_df['Date'].iloc[-1]}")
for pair in PAIRS:
    col    = pair.replace("/","_") + "_Close"
    n_null = merged_df[col].isnull().sum() if col in merged_df else "N/A"
    pct    = (n_null / len(merged_df) * 100) if isinstance(n_null, int) else 0
    print(f"  {pair:<12}  NaN in merged: {n_null:>4} / "
          f"{len(merged_df)}  ({pct:.1f}%)")
print(f"\n✅ Saved: {OUTPUT_FILE}")
print(f"   Sheets: {[s.title for s in wb.worksheets]}")

  Sheet 'BTC/USD' written — 2243 rows
  Sheet 'ETH/USD' written — 2243 rows
  Sheet 'USDC/USD' written — 1801 rows
  Sheet 'USDT/USD' written — 2243 rows

Merged shape : (2243, 21)
Date range   : 2020-01-01 → 2026-02-20
  BTC/USD       NaN in merged:    0 / 2243  (0.0%)
  ETH/USD       NaN in merged:    0 / 2243  (0.0%)
  USDC/USD      NaN in merged:  442 / 2243  (0.0%)
  USDT/USD      NaN in merged:    0 / 2243  (0.0%)

✅ Saved: ohlcv_final.xlsx
   Sheets: ['Legend', 'BTC_USD', 'ETH_USD', 'USDC_USD', 'USDT_USD', 'Final Data']
